# EDA — Modelo de predicción Mundial 2026

Análisis exploratorio del dataset que alimenta el modelo de clasificación
multiclase `[Away Win, Draw, Home Win]` y la simulación Monte Carlo del
torneo. Las interpretaciones de cada sección están basadas en los números
concretos producidos por las celdas; no se asumen patrones genéricos.

**Estructura**
1. Setup
2. Inventario de fuentes
3. Calidad de datos
4. Distribución del target
5. Análisis univariado por feature
6. Correlaciones (features ↔ features, features ↔ target)
7. ELO histórico — sanity check y comparación con ranking FIFA
8. SHAP — importancia agregada
9. Limitaciones cuantificadas del dataset
10. Conclusiones


## 1. Setup


In [ ]:
import os
import sys
from pathlib import Path

# CWD = project root
_repo_root = Path.cwd()
if _repo_root.name == "notebooks":
    os.chdir(_repo_root.parent)
    _repo_root = Path.cwd()
sys.path.insert(0, str(_repo_root))

print(f"CWD: {_repo_root}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_palette("colorblind")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"

FIG_DIR = Path("reports") / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


def savefig(name: str):
    out = FIG_DIR / f"eda_{name}.png"
    plt.savefig(out)
    print(f"  -> {out}")


from src.models.train import FEATURE_COLS
from src.features.time_decay import REFERENCE_DATE, lambda_to_halflife_years
print(f"FEATURE_COLS = {FEATURE_COLS}")
print(f"REFERENCE_DATE = {REFERENCE_DATE.date()}")


## 2. Inventario de fuentes de datos

Cinco fuentes alimentan el pipeline. Documentamos volumetría, rango temporal
y cobertura de equipos para entender qué tan informativa puede ser cada una.


In [ ]:
from src.data.data_loader import (
    load_international_results, load_fifa_ranking, load_wc2026_fixture,
    filter_relevant_matches,
)
from src.data.scraper import (
    get_statsbomb_xg_by_team, get_squad_values, SQUAD_VALUES_SNAPSHOT_DATE,
)

# 1. International results (martj42)
results_raw = load_international_results()
print(f"international_results: {len(results_raw):,} filas, "
      f"{results_raw['date'].dt.year.min()}-{results_raw['date'].dt.year.max()}")

# 2. FIFA ranking
fifa = load_fifa_ranking()
print(f"fifa_ranking: {len(fifa):,} filas, {fifa['team'].nunique()} equipos únicos")

# 3. Fixture WC 2026
fixture = load_wc2026_fixture()
print(f"wc2026_fixture: {len(fixture)} equipos en {fixture['group'].nunique()} grupos")

# 4. xG StatsBomb
try:
    xg = get_statsbomb_xg_by_team()
    print(f"statsbomb_xg: {len(xg)} equipos con xG")
except Exception as e:
    print(f"statsbomb_xg no disponible: {e}")
    xg = pd.DataFrame()

# 5. Squad values (snapshot manual)
squad = get_squad_values()
print(f"squad_values: {len(squad)} equipos (snapshot {SQUAD_VALUES_SNAPSHOT_DATE})")


In [ ]:
inventory = pd.DataFrame([
    {"fuente": "international_results", "filas": len(results_raw),
     "rango": f"{results_raw['date'].dt.year.min()}-{results_raw['date'].dt.year.max()}",
     "equipos": pd.concat([results_raw['home_team'], results_raw['away_team']]).nunique()},
    {"fuente": "fifa_ranking", "filas": len(fifa), "rango": "1992-2024", "equipos": fifa['team'].nunique()},
    {"fuente": "wc2026_fixture", "filas": len(fixture), "rango": "2026", "equipos": fixture['team'].nunique()},
    {"fuente": "statsbomb_xg", "filas": len(xg), "rango": "UEFA/FIFA OpenData", "equipos": len(xg)},
    {"fuente": "squad_values", "filas": len(squad), "rango": SQUAD_VALUES_SNAPSHOT_DATE, "equipos": len(squad)},
])
inventory


**Observaciones de la tabla.**

- `international_results` aporta el grueso del volumen (~49 k partidos, 336
  selecciones) y es la única fuente con datos pre-1990 — sirve como base para
  calcular ELO histórico desde 1872.
- `fifa_ranking` (67 k filas, 216 equipos) cubre solo 1992–2024; antes no
  existía ranking oficial, lo cual es coherente con la fecha de creación del
  sistema FIFA.
- `statsbomb_xg` cubre 109 equipos — un orden de magnitud menos que
  `international_results`. Esto adelanta una limitación: los xG son escasos
  para confederaciones AFC, CAF y CONCACAF (se cuantifica en la sección 9).
- `squad_values` cubre 62 equipos (snapshot 2026-05). Coincidirá con los 48
  del Mundial, pero deja huecos para selecciones históricas usadas en
  entrenamiento.


## 3. Calidad de datos


In [ ]:
from src.features.features import data_quality_report
data_quality_report(results_raw, "international_results crudo")


In [ ]:
# Filtrado al subset que usa el modelo
results = filter_relevant_matches(results_raw, year_cutoff=1990)
data_quality_report(results, "international_results filtrado")


**Lectura del filtrado.**

- Crudo: 49,329 partidos, 1872-2026, **72 nulos** en `home_score`/`away_score`
  (partidos pendientes en el año en curso) y **0 duplicados exactos**.
- Tras filtrar por torneos relevantes y `year_cutoff=1990`: quedan **12,640
  partidos** y **219 equipos** distintos. La reducción es drástica (~74%)
  porque se descarta el grueso de amistosos pre-1990 y partidos de torneos
  menores que el modelo no usa.
- Los 72 nulos siguen presentes y serán removidos en `build_match_features`
  por el `dropna(subset=['home_score', 'away_score'])`.


In [ ]:
# Distribución por torneo (top 15)
top_tournaments = results['tournament'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 6))
top_tournaments.plot(kind='barh', ax=ax)
ax.invert_yaxis()
ax.set_xlabel("# partidos")
ax.set_title("Top-15 torneos en el dataset filtrado")
plt.tight_layout()
savefig("03_top_tournaments")
plt.show()


In [ ]:
# Distribución de partidos por año
fig, ax = plt.subplots(figsize=(12, 5))
results.groupby(results['date'].dt.year).size().plot(ax=ax, color='steelblue')
ax.set_xlabel("Año")
ax.set_ylabel("# partidos")
ax.set_title("Volumen de partidos por año (1990-)")
plt.tight_layout()
savefig("03_matches_per_year")
plt.show()


In [ ]:
# Cobertura por equipo
team_counts = pd.concat([results['home_team'], results['away_team']]).value_counts()
print(f"Total equipos en el dataset filtrado: {len(team_counts)}")
print(f"\nTop-10 equipos con más partidos:")
print(team_counts.head(10).to_string())
print(f"\nBottom-10 equipos con menos partidos:")
print(team_counts.tail(10).to_string())


**Cobertura por equipo (datos observados).**

- Los 10 más activos (Spain 289, Italy 284, Argentina 281, Brazil 276,
  Germany 274, …) tienen entre 260 y 290 partidos relevantes en 30 años —
  ~9 por año, consistente con un calendario UEFA / Eliminatorias + Eurocopas
  / Mundial.
- En el otro extremo, las selecciones pequeñas (Eritrea 10, Yugoslavia 13,
  American Samoa 15, Vanuatu 17, …) tienen muestras de tamaño insuficiente
  para estimación ELO confiable. Cualquier predicción del modelo sobre estos
  equipos hereda un ELO cercano al `INITIAL_RATING=1500`.
- Esto motiva la limitación documentada en el README sobre debutantes y se
  reanaliza en la sección 9 restringiendo a equipos del WC 2026.


## 4. Distribución del target


In [ ]:
features = pd.read_csv("data/processed/features.csv", parse_dates=["date"])
print(f"features.csv: {len(features):,} filas, columnas {list(features.columns)}")
features.head()


In [ ]:
target_map = {0: "Away Win", 1: "Draw", 2: "Home Win"}
target_counts = features['target'].value_counts().sort_index()
target_pct = (target_counts / target_counts.sum() * 100).round(2)
dist = pd.DataFrame({"clase": [target_map[i] for i in target_counts.index],
                     "count": target_counts.values, "pct": target_pct.values})
print(dist.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(data=dist, x='clase', y='pct', ax=ax)
for i, v in enumerate(dist['pct']):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha='center')
ax.set_ylabel("% del dataset")
ax.set_title("Distribución del target")
plt.tight_layout()
savefig("04_target_distribution")
plt.show()


**Desbalance observado.**

Sobre 12,157 partidos con features completas:

| Clase    | Conteo | %      |
|----------|--------|--------|
| Home Win | 5,893  | 48.47% |
| Away Win | 3,722  | 30.62% |
| Draw     | 2,542  | 20.91% |

**Consecuencias para el modelado.**

1. Home Win representa casi la mitad del dataset: un clasificador que prediga
   "Home Win" siempre lograría accuracy ≈ 48% sin aprender nada. Por eso la
   rúbrica exige Log-Loss y Brier en lugar de accuracy.
2. Draw es la clase minoritaria con un factor 2.3× respecto a Home Win →
   sin `class_weight="balanced"` el modelo tiende a infra-predecir empates,
   que son justo los partidos más informativos para el Monte Carlo
   (50/50 → penales en eliminatorias).
3. La proporción Away/Draw ≈ 1.46 sugiere que la "localía" del primer
   equipo del par es real incluso en torneos donde el campo es neutral.
   El sesgo home/away que la simulación corrige (sección 4 del refactor
   anterior) sería todavía peor si no se promediara `predict_fn` con su
   inversa.


## 5. Análisis univariado por feature

Las celdas reutilizan `plot_feature_distribution` para mostrar histograma KDE
por clase + boxplot. Los comentarios citan los descriptivos numéricos
producidos por `df.describe()`.


In [ ]:
def plot_feature_distribution(df, feature, name_slug):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    df_plot = df.copy()
    df_plot['Resultado'] = df_plot['target'].map(target_map)

    sns.histplot(data=df_plot, x=feature, hue='Resultado',
                 kde=True, bins=40, ax=axes[0], alpha=0.5)
    axes[0].set_title(f"Distribución de {feature} por resultado")

    sns.boxplot(data=df_plot, x='Resultado', y=feature,
                order=["Away Win", "Draw", "Home Win"], ax=axes[1])
    axes[1].set_title(f"{feature} por clase (boxplot)")

    plt.tight_layout()
    savefig(f"05_{name_slug}")
    plt.show()
    print(df[feature].describe().round(3).to_string())


### 5.1 `elo_diff`

In [ ]:
plot_feature_distribution(features, 'elo_diff', 'elo_diff')

**Interpretación.**

- Rango observado [-1058, 1066] con std=229.6 y mediana=4.7 — distribución
  aproximadamente simétrica alrededor de 0 (como debería ser, ya que el
  campo `home_team`/`away_team` no implica localía universal en este
  dataset).
- Las tres KDE están **claramente separadas**: el pico de Away Win se
  centra en ≈ -150, el de Draw en ≈ 0, el de Home Win en ≈ +150. Es la
  separación más limpia de las 7 features.
- Los boxplots refuerzan: mediana Away Win ≈ -150, Draw ≈ -25, Home Win
  ≈ +120. El solapamiento es alto (las distribuciones se cruzan) pero la
  señal monotónica con el resultado es robusta.
- Esta lectura visual coincide con la correlación point-biserial calculada
  en la sección 6 (|r| ≈ 0.48-0.50, la más alta del conjunto).


### 5.2 `squad_value_diff`

In [ ]:
plot_feature_distribution(features, 'squad_value_diff', 'squad_value_diff')

**Interpretación.**

- Rango [-5.14, 5.14] en escala logarítmica, mediana = 0. El histograma
  muestra un **pico extremo en cero**: muchos pares de equipos comparten
  el `default_val` (mediana del dataset) porque no tienen entrada en
  `_SQUAD_VALUES_EUR`. Esto era predecible — el snapshot cubre 62 equipos
  pero el set de entrenamiento incluye 219.
- La separación entre clases es **modesta pero no nula**: las medianas
  por clase difieren (boxplot muestra Home Win desplazado a la derecha).
  Cuantitativamente esto se traduce en |r| ≈ 0.14 con el target —
  alrededor de 3× menos señal que ELO.
- Recomendación: dejar como feature porque aporta información ortogonal
  (capacidad económica vs. forma deportiva), pero no esperar que el
  modelo le asigne peso alto.


### 5.3 `xg_avg_for`

In [ ]:
plot_feature_distribution(features, 'xg_avg_for', 'xg_avg_for')

**Interpretación.**

- Rango [-2.35, 2.35], mediana=0, std=0.56. El pico en cero proviene de
  los equipos sin xG real en StatsBomb que reciben el `fillna(1.2)` (la
  resta de dos defaults = 0). Esto inflan la masa central artificialmente.
- Boxplots: separación visible pero pequeña — mediana Home Win ligeramente
  positiva, Away Win ligeramente negativa.
- Correlación point-biserial |r| ≈ 0.16, ligeramente mejor que squad_value
  pero todavía un orden de magnitud por debajo de ELO. La cobertura real
  (sección 9) limita su utilidad.


### 5.4 `xg_avg_against`

In [ ]:
plot_feature_distribution(features, 'xg_avg_against', 'xg_avg_against')

**Interpretación.**

- Mismo patrón que `xg_avg_for` pero con menor amplitud (std=0.50) y
  separación entre clases todavía más débil. |r| con el target ≈ 0.09.
- En la práctica, `xg_avg_against` aporta poca información adicional sobre
  el modelo (lo confirma SHAP en sección 8). La razón probable: el xG
  defensivo en partidos amistosos es muy ruidoso y los equipos sin datos
  reales dominan la masa central.


### 5.5 `travel_distance_home` y `travel_distance_away`

In [ ]:
print("travel_distance_home == 0 (campo neutral / sin geocoding):",
      f"{(features['travel_distance_home'] == 0).mean()*100:.1f}%")
print("travel_distance_away == 0:",
      f"{(features['travel_distance_away'] == 0).mean()*100:.1f}%")
plot_feature_distribution(features, 'travel_distance_home', 'travel_distance_home')


**Hallazgo crítico — `travel_distance_home` es casi siempre cero.**

- **97.7% de los partidos tienen `travel_distance_home = 0`**. Esto se
  debe a la convención de la fuente: en `international_results`, el campo
  `country` suele coincidir con el país del `home_team`, por lo que el
  equipo local no viaja. El feature es entonces casi una constante.
- El boxplot lo confirma: las cajas de las tres clases están aplastadas
  contra cero; los pocos outliers (5,000-17,500 km) corresponden a partidos
  jugados en sede neutral lejana (Mundiales, Copa América en host externo).
- **Implicación**: este feature **no aporta señal univariada al modelo**
  (|r| ≈ 0.02 — efectivamente ruido). El SHAP de la sección 8 lo
  confirma colocándolo último en importancia (≈ 0). Sin embargo, no se
  retira porque la convención del modelo requiere las 7 features para
  consistencia con `team_features` durante la simulación, donde sí se
  calcula `host_distance` significativo.


In [ ]:
plot_feature_distribution(features, 'travel_distance_away', 'travel_distance_away')

**Interpretación.**

- A diferencia del local, el visitante sí viaja: mediana=1,673 km,
  std=2,435 km, máximo=18,987 km (antípodas, p.ej. Nueva Zelanda vs.
  selección europea en Sudamérica). Solo el 18.7% son 0 (partidos donde
  el "visitante" en el CSV coincide con el país de la sede).
- Los boxplots muestran que las medianas por clase son **casi idénticas**
  (~1,600-1,800 km en las 3 clases). La distribución es derecha-asimétrica
  para todos por igual.
- Univariadamente la señal es débil (|r| ≈ 0.06), pero SHAP la ubica como
  3ª feature más importante (sección 8). Esto sugiere que el modelo la
  usa **en interacción** con ELO o xG (p.ej. distinguir entre "equipo
  fuerte en casa" vs "equipo fuerte tras viaje largo").


### 5.6 `ranking_diff`

In [ ]:
plot_feature_distribution(features, 'ranking_diff', 'ranking_diff')

**Interpretación.**

- Rango [-210, 209] (216 equipos en el ranking FIFA → diferencia teórica
  máxima 215), std=60.9, mediana=2 → distribución casi simétrica.
- Las tres KDE están separadas con el mismo patrón que ELO: Away Win
  desplazado a la izquierda, Home Win a la derecha. Boxplots confirman.
- |r| ≈ 0.47-0.49, prácticamente igual a `elo_diff`. La sección 6 mostrará
  que ambos features están correlacionados (r = 0.84), lo que **abre la
  pregunta**: ¿es redundante mantener los dos? La respuesta queda
  diferida a SHAP — si el modelo asigna importancia significativa a ambos,
  hay información complementaria; si solo a uno, se podría eliminar el
  otro.


### 5.7 `time_weight` — decay exponencial

In [ ]:
halflife = lambda_to_halflife_years(0.002)
print(f"Half-life del decay con lambda=0.002: {halflife:.2f} años")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(features['time_weight'], bins=40, color='teal', alpha=0.7)
axes[0].set_title("Distribución de time_weight")
axes[0].set_xlabel("time_weight")

# Serie temporal
sample = features.sort_values('date').iloc[::max(1, len(features)//500)]
axes[1].plot(sample['date'], sample['time_weight'], color='teal')
axes[1].set_title(f"time_weight vs fecha (half-life = {halflife:.2f} años)")
axes[1].set_xlabel("Fecha del partido")
axes[1].set_ylabel("Peso")

plt.tight_layout()
savefig("05_time_weight")
plt.show()


**Análisis del decay.**

- Half-life calculado: **0.95 años**. Significa que un partido de hace 1
  año pesa la mitad que uno reciente; un partido de hace 5 años pesa
  ≈ 3% del peso de uno reciente.
- El histograma muestra una **acumulación masiva cerca de cero**:
  prácticamente todos los partidos previos a ~2022 reciben peso ≈ 0.
  Solo los últimos ~2 años contribuyen sustancialmente al entrenamiento.
- La serie temporal muestra el crecimiento exponencial desde 0 hasta 1
  acercándose a `REFERENCE_DATE = 2026-01-01`. Los partidos posteriores a
  esa fecha (los hay, hasta 2026-06) reciben peso 1.0 por el `clip(lower=0)`.
- **Implicación**: λ = 0.002 es agresivo. Está bien para reflejar cambios
  rápidos en plantillas (rotación generacional, cambios de DT) pero el
  precio es perder señal histórica. Si el modelo se queda corto en log-loss
  habría que revisar λ a la baja (p.ej. 0.001 → half-life 1.9 años).


## 6. Correlaciones


In [ ]:
corr = features[FEATURE_COLS].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Matriz de correlación entre features")
plt.tight_layout()
savefig("06_correlation_matrix")
plt.show()


**Hallazgos del heatmap (valores reales).**

| Par | r | Lectura |
|---|---|---|
| `elo_diff` ↔ `ranking_diff` | **0.84** | Alta colinealidad — esperada porque ambos miden fuerza relativa. |
| `squad_value_diff` ↔ `xg_avg_for` | **0.49** | Equipos con plantillas más valiosas generan más xG ofensivo. |
| `travel_distance_home` ↔ `travel_distance_away` | **0.35** | Los partidos lejos del local también suelen serlo del visitante (sedes neutrales). |
| `elo_diff` ↔ `squad_value_diff` | 0.28 | Correlación moderada — ELO captura algo de calidad de plantilla. |
| `elo_diff` ↔ `xg_avg_for` | 0.29 | Lo mismo, con xG. |
| `xg_avg_for` ↔ `xg_avg_against` | -0.09 | Independencia entre ofensiva y defensiva — la xG defensiva es ruidosa. |
| `travel_distance_home` ↔ resto | < 0.06 | Ortogonal a casi todo — coherente con que sea casi constante (97.7% en 0). |

**Decisión sobre `elo_diff` vs `ranking_diff`**: la colinealidad de 0.84 es
alta pero **no estricta** (1.0). Mantenerlos ambos es defendible porque ELO
incorpora margen de victoria y K-factor por torneo, mientras que el ranking
FIFA es más conservador (sección 7 lo evidencia con divergencias en el
top-10). La decisión final se valida con SHAP en la sección 8: si ambos
mantienen importancia no trivial → no son redundantes.


In [ ]:
# Correlación con target (point-biserial vs cada clase)
from scipy.stats import pointbiserialr
rows = []
for cls in [0, 1, 2]:
    y = (features['target'] == cls).astype(int)
    for feat in FEATURE_COLS:
        r, p = pointbiserialr(features[feat].fillna(0), y)
        rows.append({"feature": feat, "clase": target_map[cls],
                     "r": round(r, 3), "p_value": round(p, 5)})
corr_target = pd.DataFrame(rows).pivot(index='feature', columns='clase', values='r')
corr_target


**Lectura del point-biserial features↔target.**

Ranking de features por |r| máximo con cualquier clase (valores observados):

1. `elo_diff` — |r| hasta **0.496** (clase Home Win).
2. `ranking_diff` — |r| hasta **0.487**.
3. `xg_avg_for` — 0.159.
4. `squad_value_diff` — 0.143.
5. `xg_avg_against` — 0.093.
6. `travel_distance_away` — 0.070.
7. `travel_distance_home` — **0.025** (prácticamente ruido).

Patrón consistente en todas las features: la clase Draw recibe |r| ≈ 0.01-0.07
(menor que las otras dos). Esto significa que las features actuales **no
discriminan bien empates** — son útiles para Home/Away pero el empate
queda subdeterminado. Es información relevante para el reporte: la rúbrica
penaliza la mala calibración en partidos cerrados, y el log-loss de Draw
será sistemáticamente peor que el de las otras clases.


In [ ]:
# Pairplot reducido (sample para velocidad)
sample = features.sample(min(2000, len(features)), random_state=42).copy()
sample['Resultado'] = sample['target'].map(target_map)
selected = ['elo_diff', 'squad_value_diff', 'ranking_diff', 'xg_avg_for']
g = sns.pairplot(sample[selected + ['Resultado']], hue='Resultado',
                 diag_kind='kde', height=2.5, plot_kws={"alpha": 0.4, "s": 12})
g.fig.suptitle("Pairplot de las 4 features más informativas (n=2000)", y=1.01)
savefig("06_pairplot")
plt.show()


**Pairplot.** El scatter `elo_diff` vs `ranking_diff` muestra una
nube alineada con la diagonal — visualización directa del r=0.84. Los
gradientes de color (Away→Draw→Home) son monotónicos en ambos ejes, lo
que confirma que cualquiera de los dos sirve como predictor primario.
Las otras combinaciones (`squad_value_diff`, `xg_avg_for`) muestran nubes
más difusas con separación de clases mucho más sutil.


## 7. ELO histórico — sanity check


In [ ]:
from src.features.elo import calculate_elo_ratings

elo_df = calculate_elo_ratings(results.sort_values("date"))
print(f"ELO records: {len(elo_df):,}")
elo_df.tail(5)


In [ ]:
# ELO en el tiempo para 5 selecciones icónicas
icons = ["Brazil", "Germany", "Argentina", "Spain", "France"]
fig, ax = plt.subplots(figsize=(12, 6))
for team in icons:
    h = elo_df[elo_df['home_team'] == team][['date', 'home_elo_after']].rename(
        columns={'home_elo_after': 'elo'})
    a = elo_df[elo_df['away_team'] == team][['date', 'away_elo_after']].rename(
        columns={'away_elo_after': 'elo'})
    series = pd.concat([h, a]).sort_values('date')
    if not series.empty:
        ax.plot(series['date'], series['elo'].rolling(20).mean(), label=team, linewidth=2)
ax.legend()
ax.set_title("ELO suavizado (rolling 20) - top selecciones, 1990-")
ax.set_xlabel("Fecha"); ax.set_ylabel("ELO")
plt.tight_layout()
savefig("07_elo_history")
plt.show()


**Sanity check del ELO histórico — trayectorias observadas.**

- **España** sube fuerte desde 2008 (gana Eurocopa 2008, Mundial 2010,
  Eurocopa 2012), pico ~2014 en ~2020 ELO, baja tras la eliminación
  temprana del Mundial 2014. Pico secundario en 2022-2024 (vuelve a
  campeonatos). El modelo recoge correctamente la era dorada.
- **Argentina** muestra un valle ~2008-2014, recuperación 2014-2018, otro
  valle 2019, y subida pronunciada 2020-2022 culminando en el Mundial 2022.
- **Alemania** mantiene un perfil alto y estable 2000-2014, con pico en
  el Mundial 2014, y caída sostenida desde 2018 (eliminación en fase de
  grupos) — el modelo refleja la crisis generacional alemana.
- **Francia** tiene un pico claro ~2018 (Mundial), valle 2008-2012, y
  subida sostenida desde 2014.
- **Brasil** se mantiene en 1900-2000 ELO la mayor parte del tiempo,
  sin las oscilaciones extremas de Argentina/España.

Las trayectorias coinciden con la memoria futbolística estándar — el ELO
no muestra anomalías que sugieran bugs en `calculate_elo_ratings`.


In [ ]:
# Top-10 ELO al REFERENCE_DATE
home_last = elo_df.groupby("home_team")["home_elo_after"].last()
away_last = elo_df.groupby("away_team")["away_elo_after"].last()
last_elo = pd.concat([home_last.rename("elo"), away_last.rename("elo")])
last_elo = last_elo.groupby(last_elo.index).last().sort_values(ascending=False)
top10_elo = last_elo.head(10).round(0).reset_index()
top10_elo.columns = ["team", "elo"]
print("Top-10 ELO al final del dataset:")
print(top10_elo.to_string(index=False))


In [ ]:
# Top-10 FIFA ranking al REFERENCE_DATE
fifa_at_ref = fifa[fifa['rank_date'] <= REFERENCE_DATE].sort_values('rank_date')
latest_fifa = fifa_at_ref.groupby('team').last().sort_values('rank').head(10)
print("Top-10 ranking FIFA al REFERENCE_DATE:")
print(latest_fifa[['rank', 'total_points']].to_string())


**ELO vs FIFA al REFERENCE_DATE — divergencias observadas.**

| Posición | Top-10 ELO | Top-10 FIFA |
|---:|---|---|
| 1 | Spain (2044) | Argentina |
| 2 | Argentina (1946) | France |
| 3 | France (1937) | Belgium |
| 4 | England (1927) | Brazil |
| 5 | Japan (1882) | England |
| 6 | Australia (1863) | Portugal |
| 7 | Iran (1861) | Netherlands |
| 8 | Morocco (1859) | Spain |
| 9 | Brazil (1846) | Croatia |
| 10 | Portugal (1844) | Italy |

**Las dos listas tienen una intersección de 6 equipos** (Spain, Argentina,
France, England, Brazil, Portugal) y **4 divergencias por lado**:

- ELO incluye y FIFA no: **Japan, Australia, Iran, Morocco** — selecciones
  no-UEFA con resultados recientes fuertes (Marruecos 4° en Qatar 2022,
  Japón ganando a Alemania/España). FIFA pondera más los partidos UEFA
  por su sistema de puntos.
- FIFA incluye y ELO no: **Belgium, Netherlands, Croatia, Italy**.
  Bélgica e Italia perdieron forma reciente pero conservan inercia en
  el ranking. Italia, además, **no se clasificó al WC2026** — su presencia
  en el top-10 FIFA es informacionalmente irrelevante para el modelo.

**Conclusión**: ELO y ranking FIFA son **complementarios, no redundantes**
pese a su correlación de 0.84. ELO reacciona más rápido a forma reciente y
no penaliza a equipos no-UEFA. Mantener ambas features está justificado.


## 8. SHAP — importancia agregada de features


In [ ]:
from pathlib import Path
model_path = Path("data/processed/models/xgboost.joblib")

if model_path.exists():
    from src.models.train import load_model
    model = load_model("xgboost")
    print("Modelo XGBoost cargado.")
else:
    model = None
    print("Modelo no disponible — ejecuta `make train` primero. Saltando SHAP.")


In [ ]:
if model is not None:
    import shap
    X_sample = features[FEATURE_COLS].sample(2000, random_state=42).astype(np.float32)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer(X_sample, check_additivity=False)
    shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS, show=False)
    plt.tight_layout()
    savefig("08_shap_summary")
    plt.show()
else:
    print("(omitido)")


**Ranking SHAP observado (mean |SHAP value|, escala relativa).**

| # | Feature | Importancia | Lectura |
|---:|---|---:|---|
| 1 | `elo_diff` | **~1.00** | Dominante en las 3 clases. |
| 2 | `ranking_diff` | ~0.65 | Confirma que **no es redundante con ELO** pese al r=0.84. |
| 3 | `travel_distance_away` | ~0.30 | Útil sobre todo en interacciones (univariadamente |r|=0.07). |
| 4 | `xg_avg_for` | ~0.20 | Aporte modesto pero estable. |
| 5 | `xg_avg_against` | ~0.18 | Similar a `xg_avg_for`. |
| 6 | `squad_value_diff` | ~0.15 | Menor de lo esperado — el snapshot manual y la mediana fallback diluyen la señal. |
| 7 | `travel_distance_home` | **~0.00** | Confirma que **no aporta** por la masa de ceros (97.7%). |

**Decisiones derivadas:**

1. **No eliminar `ranking_diff`**: su importancia SHAP de ~0.65 demuestra
   que el modelo lo usa para corregir/complementar `elo_diff` — el
   pairwise r=0.84 oculta la información ortogonal que aporta.
2. **`travel_distance_home` se puede congelar a 0**: es ruido constante.
   Sin embargo, se mantiene por consistencia con el flujo de simulación
   donde sí se calcula `host_distance` no-trivial.
3. **xG y squad_value son features de "segunda línea"**: aportan pero
   no son decisivos. El reporte debe contextualizar que su efecto está
   limitado por la cobertura (sección 9).
4. **Class 0 (Away Win) y Class 2 (Home Win) reciben aportes simétricos
   de elo_diff** — esperable, son los lados opuestos de la misma señal.
   Class 1 (Draw) recibe menos contribución de ELO, lo que reconfirma
   que los empates son la clase más difícil de predecir.


## 9. Limitaciones cuantificadas del dataset


In [ ]:
from src.simulation.tournament import GROUPS_2026, ALL_TEAMS

# Partidos por equipo, restringido a equipos del WC 2026
wc26_counts = team_counts.reindex(ALL_TEAMS).fillna(0).astype(int)
print("Equipos WC 2026 con MENOS partidos históricos (post-1990):")
print(wc26_counts.sort_values().head(10).to_string())
print(f"\nMediana: {int(wc26_counts.median())} partidos")


In [ ]:
# Cobertura xG sobre equipos del Mundial
xg_teams = set(xg['team']) if not xg.empty else set()
in_xg = [t for t in ALL_TEAMS if t in xg_teams]
miss_xg = [t for t in ALL_TEAMS if t not in xg_teams]
print(f"Cobertura xG StatsBomb sobre 48 equipos WC2026: {len(in_xg)}/48 = {len(in_xg)/48*100:.1f}%")
print(f"  Equipos sin xG real (usan 1.2 default): {miss_xg[:15]}{' ...' if len(miss_xg) > 15 else ''}")


In [ ]:
# Cobertura squad_value
sv_teams = set(squad['team'])
in_sv = [t for t in ALL_TEAMS if t in sv_teams]
miss_sv = [t for t in ALL_TEAMS if t not in sv_teams]
print(f"Cobertura squad_value sobre 48 equipos WC2026: {len(in_sv)}/48 = {len(in_sv)/48*100:.1f}%")
print(f"  Equipos sin valor (usan mediana): {miss_sv}")


**Limitaciones numéricas medidas (no asumidas).**

**1. Equipos del WC2026 con escaso historial post-1990 (mediana del grupo = 213):**

| Equipo | # partidos | Δ vs mediana |
|---|---:|---:|
| Cape Verde | 53 | -160 |
| New Zealand | 64 | -149 |
| Curacao | 67 | -146 |
| DR Congo | 78 | -135 |
| Senegal | 85 | -128 |
| South Africa | 86 | -127 |
| Haiti | 86 | -127 |
| Ivory Coast | 88 | -125 |
| Algeria | 89 | -124 |
| Egypt | 94 | -119 |

Para estos 10 equipos el ELO estimado tiene mayor varianza (menos datos
para revertir desde el `INITIAL_RATING=1500`). Cabe esperar predicciones
con peor calibración cuando alguno juegue.

**2. Cobertura xG StatsBomb sobre los 48 del Mundial: 35/48 = 72.9%.**

Los **13 equipos sin xG real** (reciben `xg_for = xg_against = 1.2` por
default) son:

South Africa, Bosnia & Herzegovina, Haiti, Curacao, Ivory Coast, New Zealand,
Cape Verde, Iraq, Norway, Algeria, Jordan, DR Congo, Uzbekistan.

Coincide en gran medida con la lista de "pocos partidos". Para estos
equipos los features `xg_avg_for` y `xg_avg_against` son **ruido**
respecto al rival, no señal. El SHAP los ubicó en 4° y 5° lugar
globalmente, pero su contribución para este subgrupo es nula.

**3. Cobertura `squad_value`: 48/48 = 100%.** El snapshot manual del
scraper cubre todos los equipos clasificados, así que esta feature no
sufre el problema de cobertura desigual.

**4. Distorsión del feature `travel_distance_home`** (sección 5.5): 97.7%
de los partidos lo tienen en 0. Esto no es un bug, es la consecuencia de
que el dataset histórico raras veces se juega fuera del país del local.
SHAP lo confirmó como inútil. Para la simulación del WC 2026 el equivalente
es `host_distance` en `team_features`, que sí es informativo (distancia a
sedes USA/MEX/CAN).


## 10. Conclusiones

**Hallazgos cuantitativos para citar en el reporte técnico:**

1. **Volumen y filtrado.** 49,329 partidos crudos → 12,640 tras filtrar
   por torneos relevantes y `year >= 1990` → 12,157 con features
   completas tras `dropna`. 219 equipos en entrenamiento, 48 en simulación.

2. **Desbalance del target.** Home 48.47% / Away 30.62% / Draw 20.91%.
   El factor 2.3× entre clases justifica `class_weight="balanced"` y
   métricas log-loss/Brier sobre accuracy.

3. **Features con mayor poder discriminativo univariado**
   (|r| point-biserial vs target):

   ```
   elo_diff             0.496
   ranking_diff         0.487
   xg_avg_for           0.159
   squad_value_diff     0.143
   xg_avg_against       0.093
   travel_distance_away 0.070
   travel_distance_home 0.025
   ```

4. **Colinealidad relevante.** `elo_diff` ↔ `ranking_diff` r=0.84.
   SHAP confirma que **no son redundantes**: la 2ª feature mantiene
   importancia ~0.65 (frente a 1.0 de la 1ª).

5. **Decay temporal agresivo.** λ=0.002 → half-life 0.95 años. El 97% de
   los partidos del dataset entran al entrenamiento con peso < 0.1.
   Decisión defendible pero acotada — λ menor podría mejorar la log-loss.

6. **Empates infra-discriminados.** Todas las features tienen |r| < 0.07
   con la clase Draw. El modelo tendrá peor log-loss para empates que
   para victorias.

7. **Limitaciones cuantificadas.**
   - 13/48 (27.1%) equipos del WC2026 sin xG real.
   - 10/48 (~21%) con < 100 partidos históricos en 30 años.
   - `travel_distance_home` univariadamente inútil (97.7% en 0).

Las 15 figuras de `reports/figures/eda_*.png` documentan visualmente cada
uno de estos puntos y son aptas para reutilizarse en el reporte IEEE.
